In [ ]:
import matplotlib.pyplot as plt
from astropy import units as u
import pandas as pd
import os
from astropy.io import fits
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
import jax
import jax.numpy as jnp


In [ ]:
# function to find spikes in the loss
def find_spikes(loss, threshold=0.1):
    spikes = []
    for i in range(1, len(loss) - 1):
        if loss[i] > loss[i - 1] * (1 + threshold) and loss[i] > loss[i + 1] * (1 + threshold):
            spikes.append(i)
    return spikes

In [ ]:
# import data
optimisation_log = pd.read_csv("streamfit_test_output/optimisation_log.csv")
trace_log_path = "streamfit_test_output/optimisation_trace.csv"
if os.path.exists(trace_log_path):
    trace_log = pd.read_csv(trace_log_path)
    print(f"Loaded tracer log: {trace_log_path} ({len(trace_log)} rows)")
else:
    trace_log = None
    print(f"Tracer log not found at {trace_log_path}. Re-run test_streamfit.ipynb with trace_file enabled.")

optimisation_log

epochs = optimisation_log["epoch"].values
loss = optimisation_log["loss"].values

lowest_loss = min(loss)
best_epoch = epochs[loss.argmin()]

spikes = find_spikes(loss, threshold=0.1)
spike_epochs = epochs[spikes]
print(f"Spikes found at epochs: {spike_epochs.tolist()}")

plt.plot(epochs, loss)
plt.scatter(best_epoch, lowest_loss, color="green", label=f"Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}")
plt.scatter(spike_epochs, loss[spikes], color="orange", label="Spikes")
plt.legend()
plt.xlabel("Epoch")
plt.ylabel("Loss")


We want to investigate which change in parameters causes the spikes in loss

In [ ]:
# plot the parameters over epochs in a single figure with subplots
param_names = [col for col in optimisation_log.columns if col not in ("epoch", "loss")]

fig, axes = plt.subplots(len(param_names) + 1, 1, figsize=(8, 3 * (len(param_names) + 1)), sharex=True)
loss_ax = axes[0]
loss_ax.plot(epochs, loss, color="black")
loss_ax.scatter(best_epoch, lowest_loss, color="green", label=f"Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}")
loss_ax.scatter(epochs[spikes], loss[spikes], color="orange", label="Spikes")
loss_ax.set_ylabel("loss")
loss_ax.set_yscale("log")
loss_ax.grid(True)
loss_ax.legend()

for ax, param in zip(axes[1:], param_names):
    param_values = optimisation_log[param].values
    ax.plot(epochs, param_values)
    ax.scatter(best_epoch, param_values[loss.argmin()], color="green", label="Best Epoch")
    ax.scatter(epochs[spikes], param_values[spikes], color="orange", label="Spikes")
    ax.set_ylabel(param)
    ax.grid(True)
    ax.legend()

axes[-1].set_xlabel("Epoch")
fig.tight_layout()
plt.show()

In [ ]:
# plot tracer diagnostics over epochs and mark loss-spike epochs
if trace_log is None:
    raise FileNotFoundError("No optimisation trace log found. Run test_streamfit.ipynb with trace_file enabled first.")

trace_log = trace_log.sort_values("epoch").reset_index(drop=True)
trace_cols = [col for col in trace_log.columns if col not in ("epoch", "loss")]

if not trace_cols:
    raise ValueError("No tracer columns found in optimisation trace log.")

spike_epoch_set = set(int(e) for e in spike_epochs.tolist())
trace_spike_mask = trace_log["epoch"].astype(int).isin(spike_epoch_set)

fig, axes = plt.subplots(len(trace_cols) + 1, 1, figsize=(10, 2.5 * (len(trace_cols) + 1)), sharex=True)

loss_ax = axes[0]
loss_ax.plot(epochs, loss, color="black")
loss_ax.scatter(best_epoch, lowest_loss, color="green", label=f"Best Epoch: {best_epoch}\n Loss: {lowest_loss:.0f}")
loss_ax.scatter(spike_epochs, loss[spikes], color="orange", label="Spikes")
loss_ax.set_ylabel("loss")
loss_ax.set_yscale("log")
loss_ax.grid(True, alpha=0.3)
loss_ax.legend()

for ax, col in zip(axes[1:], trace_cols):
    ax.plot(trace_log["epoch"].values, trace_log[col].values, color="tab:blue")
    if trace_spike_mask.any():
        ax.scatter(
            trace_log.loc[trace_spike_mask, "epoch"].values,
            trace_log.loc[trace_spike_mask, col].values,
            color="orange",
            s=18,
            label="Spikes",
            zorder=5,
        )
        ax.legend()
    ax.set_ylabel(col)
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Epoch")
fig.tight_layout()
plt.show()


Plot the model at every epoch

In [ ]:
# get the point cloud first
cubefile = 'test_data/HLTAU_HCOp32.fits'
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

print("Loaded cube with shape:", cube.shape)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 

n_points = 10 # the number of points we want to reduce the data to


# Extract 1D streamline from the data cube
pc_coords, pc_means, pc_stds = extract_streamline.reduce_to_1D(streamer_cube, n_elements=n_points)

# Prepare data for gradient descent
ra_data = pc_means[0] # offsets in arcsec
dec_data = pc_means[1] # offsets in arcsec
v_data = pc_means[2]   # velocities in km/s (rel to vlsr)

ra_sigma = pc_stds[0]
dec_sigma = pc_stds[1]
v_sigma = pc_stds[2]


data = (ra_data, dec_data, v_data)
uncertainties = (ra_sigma, dec_sigma, v_sigma)

In [ ]:
# Fixed parameters (not optimized)
# pa = 138 + 270 = 408 degrees, which is equivalent to 48 degrees (since PA is modulo 360)
fixed_params = {
    'mass': 2.1,  # Msun
    'inc': -47.0,  # degrees
    'pa': 48.0,  # degrees
    'rmin': 50.0,  # au
    'deltar': 40.0,  # au
    'v_lsr': 7.1  # km/s (systemic velocity)
}
distance = 147 # pc

# Convert fixed angle params to radians to match forward_model expectations
fixed_params['inc'] = float(jnp.deg2rad(fixed_params['inc']))
fixed_params['pa'] = float(jnp.deg2rad(fixed_params['pa']))

# Precompute model and matched values once per epoch for reuse
epoch_models = []
for idx, epoch in enumerate(epochs):
    row = optimisation_log.iloc[idx]
    opt_params_epoch = {param: float(row[param]) for param in param_names}

    ra_model, dec_model, v_model = gradient_descent.forward_model(opt_params_epoch, fixed_params, distance)
    not_nan = ~jnp.isnan(ra_model) & ~jnp.isnan(dec_model) & ~jnp.isnan(v_model)
    ra_model = ra_model[not_nan]
    dec_model = dec_model[not_nan]
    v_model = v_model[not_nan]

    ra_model_interp, dec_model_interp, v_model_interp, valid = gradient_descent.match_model_to_data_curve(
        ra_model, dec_model, v_model, ra_data, dec_data)

    epoch_models.append({
        'epoch': epoch,
        'opt_params_epoch': opt_params_epoch,
        'ra_model': ra_model,
        'dec_model': dec_model,
        'v_model': v_model,
        'ra_model_interp': ra_model_interp,
        'dec_model_interp': dec_model_interp,
        'v_model_interp': v_model_interp,
        'valid': valid,
        'v_data_valid': v_data[valid],
        'ra_sigma_valid': ra_sigma[valid],
        'dec_sigma_valid': dec_sigma[valid],
        'v_sigma_valid': v_sigma[valid],
    })

# Create subplots - calculate grid dimensions
num_epochs = len(epochs)
n_cols = 4
n_rows = (num_epochs + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    ra_model = epoch_data['ra_model']
    dec_model = epoch_data['dec_model']
    ra_model_interp = epoch_data['ra_model_interp']
    dec_model_interp = epoch_data['dec_model_interp']
    ra_sigma_valid = epoch_data['ra_sigma_valid']
    dec_sigma_valid = epoch_data['dec_sigma_valid']
    opt_params_epoch = epoch_data['opt_params_epoch']
    epoch = epoch_data['epoch']

    if idx == 0:
        print(opt_params_epoch)
        print(fixed_params)
        print(ra_model_interp[:5], dec_model_interp[:5])

    ax.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, label='Point cloud')
    ax.errorbar(ra_data, dec_data, xerr=ra_sigma_valid, yerr=dec_sigma_valid, fmt='o-', label='Extracted 1D Streamline', color='red')
    ax.plot(ra_model, dec_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(ra_model_interp, dec_model_interp, s=25, label='Model at data arc lengths', color='blue', zorder=5)
    ax.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.invert_xaxis()
    ax.set_xlabel('RA Offset (arcsec)')
    ax.set_ylabel('Dec Offset (arcsec)')
    ax.set_title('Current Streamline Model')
    ax.text(0.05, 0.95, f"Epoch: {int(epoch)}", transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)

fig.tight_layout()
plt.savefig("streamfit_test_output/streamline_fits_over_epochs.png", bbox_inches='tight')
plt.show()

In [ ]:
# plot ra vs velocity for every epoch in a single figure with subplots
if 'epoch_models' not in globals():
    raise RuntimeError("Run Cell 8 first to precompute epoch_models.")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    ra_model = epoch_data['ra_model']
    v_model = epoch_data['v_model']
    ra_model_interp = epoch_data['ra_model_interp']
    v_model_interp = epoch_data['v_model_interp']
    v_data_valid = epoch_data['v_data_valid']
    ra_sigma_valid = epoch_data['ra_sigma_valid']
    v_sigma_valid = epoch_data['v_sigma_valid']
    epoch = epoch_data['epoch']

    ax.scatter(pc_coords[0], pc_coords[2], s=1, alpha=0.3, label='Point cloud')
    ax.scatter(0, 7, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.errorbar(ra_data, v_data_valid, xerr=ra_sigma_valid, yerr=v_sigma_valid, fmt='o-', label='Extracted 1D Streamline', color='red')
    ax.plot(ra_model, v_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(ra_model_interp, v_model_interp, s=25, label='Model at data arc lengths', color='blue', zorder=5)
    ax.set_xlabel('RA Offset (arcsec)')
    ax.set_ylabel('Velocity (km/s)')
    ax.set_title('RA vs Velocity')
    ax.text(0.05, 0.95, f"Epoch: {int(epoch)}", transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)
fig.tight_layout()
plt.savefig("streamfit_test_output/ra_vs_velocity_over_epochs.png", bbox_inches='tight')
plt.show()

In [ ]:
# plot dec vs velocity for every epoch in a single figure with subplots
if 'epoch_models' not in globals():
    raise RuntimeError("Run Cell 8 first to precompute epoch_models.")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()
for idx, epoch_data in enumerate(epoch_models):
    ax = axes[idx]

    dec_model = epoch_data['dec_model']
    v_model = epoch_data['v_model']
    dec_model_interp = epoch_data['dec_model_interp']
    v_model_interp = epoch_data['v_model_interp']
    v_data_valid = epoch_data['v_data_valid']
    dec_sigma_valid = epoch_data['dec_sigma_valid']
    v_sigma_valid = epoch_data['v_sigma_valid']
    epoch = epoch_data['epoch']

    ax.scatter(pc_coords[1], pc_coords[2], s=1, alpha=0.3, label='Point cloud')
    ax.scatter(0, 7, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    ax.errorbar(dec_data, v_data_valid, xerr=dec_sigma_valid, yerr=v_sigma_valid, fmt='o-', label='Extracted 1D Streamline', color='red')
    ax.plot(dec_model, v_model, color='blue', linewidth=2, label='Model Streamline')
    ax.scatter(dec_model_interp, v_model_interp, s=25, label='Model at data arc lengths', color='blue', zorder=5)
    ax.set_xlabel('Dec Offset (arcsec)')
    ax.set_ylabel('Velocity (km/s)')
    ax.set_title('Dec vs Velocity')
    ax.text(0.05, 0.95, f"Epoch: {int(epoch)}", transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

for idx in range(num_epochs, len(axes)):
    axes[idx].axis('off')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=5, frameon=True)
fig.tight_layout()
plt.savefig("streamfit_test_output/dec_vs_velocity_over_epochs.png", bbox_inches='tight')
plt.show()